# Fundamentos de Ingeniería de Datos
## INGESTA: capa RAW

```
Kaggle → CSV completo (local) → muestra estratificada (CSV) → SQLite (raw_autos)
```

| Elemento | Valor |
|---|---|
| Dataset | `andreinovikov/used-cars-dataset` (anuncios de autos usados de Cars.com, EE. UU.) |
| Etiqueta | `autos` |
| Resultado | `Datasets_raw/andreinovikov_used-cars-dataset/cars_muestra.csv` y la tabla `raw_autos` en `base_datos.db` |

La elección del dataset está documentada en `BUSQUEDA.ipynb`.

### Configuración del entorno del proyecto

Importamos `utils.py`, el módulo que centraliza las rutas y funciones que usan **todos los notebooks del pipeline**.

In [1]:
# ── Importar utils.py ──────────────────────────────────────────────────────

# utils.py contiene las rutas del proyecto y funciones compartidas.
# Vive en Utilidades/utils.py, en la raíz del repositorio.

import sys
import os
sys.path.append(os.path.abspath('../Utilidades'))   # el notebook se ejecuta desde Codigos/

from utils import (
    PROYECTO_BASE, PRIVADO_PATH, DATASETS_RAW_PATH, DB_PATH,
    inspeccionar_dataset, listar_tablas, tamano_mb, ruta_relativa
)

print()
print('utils.py importado correctamente')
print(f'Proyecto base: {os.path.basename(PROYECTO_BASE)}/')
print()


utils.py importado correctamente
Proyecto base: ING_BD_ProjectF/



In [2]:
# ── Importar librerías y crear carpetas base ──────────────────────────────

import sqlite3
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

os.makedirs(DATASETS_RAW_PATH, exist_ok=True)
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

print()
print('Entorno listo')
print(f'  Datasets raw  : {ruta_relativa(DATASETS_RAW_PATH)}')
print(f'  Base de datos : {ruta_relativa(DB_PATH)}')
print()


Entorno listo
  Datasets raw  : Ingesta_de_Datos/Datasets_raw
  Base de datos : Ingesta_de_Datos/Base_de_datos/base_datos.db



### PASO 1 — Configuración de la API de Kaggle

Para autenticarte con Kaggle necesitas tu **usuario** y tu **API key**. Se obtienen en [kaggle.com](https://www.kaggle.com) → Settings → API → **Create New Token**, que descarga un archivo `kaggle.json`.

In [3]:
# ── Credenciales de Kaggle: kaggle.json en la carpeta Privado/ ────────────
# 1. Descarga tu token en kaggle.com → Settings → API → Create New Token
# 2. Cópialo a: Privado/kaggle.json  (esa carpeta está en .gitignore, nunca se sube)

KAGGLE_JSON_PATH = f'{PRIVADO_PATH}/kaggle.json'

if os.path.exists(KAGGLE_JSON_PATH):
    os.environ['KAGGLE_CONFIG_DIR'] = PRIVADO_PATH   # carpeta que contiene kaggle.json
    print()
    print('kaggle.json encontrado en Privado/ — usando esa configuración.')
    print()

else:
    # ── Alternativa: ingreso interactivo (solo vive en esta sesión) ──
    from getpass import getpass
    os.environ['KAGGLE_USERNAME'] = input('Usuario de Kaggle: ')
    os.environ['KAGGLE_KEY']      = getpass('API key de Kaggle (no se mostrará en pantalla): ')
    print()
    print('Credenciales cargadas solo en esta sesión (no se guardan en el notebook).')
    print()

import kaggle                                        # se autentica al importarse

print('Conexión con la API de Kaggle lista')
print()


kaggle.json encontrado en Privado/ — usando esa configuración.



Conexión con la API de Kaggle lista



### PASO 2 — Dataset de la fuente

Toda la configuración de la ingesta está en una sola celda. Si se cambiara de dataset, solo se editaría aquí.

| Parámetro | Valor | Por qué |
|---|---|---|
| `TAMANO_MUESTRA` | 200 000 | Suficiente para el modelo y el tablero, y cabe en GitHub (máximo 100 MB por archivo) |
| `COLUMNA_ESTRATO` | `manufacturer` | Cada marca conserva su proporción del mercado |
| `SEMILLA` | 42 | La muestra es la misma cada vez que se ejecuta el notebook |

In [4]:
# ── Configuración de la fuente ─────────────────────────────────────────────

ETIQUETA = 'autos'
REF      = 'andreinovikov/used-cars-dataset'                 # usuario/dataset en Kaggle

CARPETA_FUENTE = f'{DATASETS_RAW_PATH}/{REF.replace("/", "_")}'
CSV_COMPLETO   = f'{CARPETA_FUENTE}/cars.csv'                # 762k filas: NO se sube (.gitignore)
CSV_MUESTRA    = f'{CARPETA_FUENTE}/cars_muestra.csv'        # 200k filas: SÍ se sube

TAMANO_MUESTRA  = 200_000
COLUMNA_ESTRATO = 'manufacturer'
SEMILLA         = 42

TABLA_RAW = f'raw_{ETIQUETA}'

# Metadatos de la fuente
metadatos = pd.DataFrame([{
    'etiqueta': ETIQUETA,
    'nombre'  : 'Used Cars Dataset (Cars.com)',
    'ref'     : REF,
    'url'     : f'https://www.kaggle.com/datasets/{REF}',
    'formato' : 'CSV',
    'licencia': 'CC0-1.0 (dominio público)',
}])

metadatos

,etiqueta,nombre,ref,url,formato,licencia
0,autos,Used Cars Dataset (Cars.com),andreinovikov/used-cars-dataset,https://www.kaggle.com/datasets/andreinovikov/...,CSV,CC0-1.0 (dominio público)


### PASO 3 — Descarga del dataset completo

In [5]:
# ── Descargar el dataset (solo si no está ya en la carpeta) ────────────────

os.makedirs(CARPETA_FUENTE, exist_ok=True)

print()
if os.path.exists(CSV_COMPLETO):
    print('El CSV completo ya existe: no se vuelve a descargar.')
else:
    print(f'Descargando: {REF} ...')
    kaggle.api.dataset_download_files(REF, path=CARPETA_FUENTE, unzip=True, quiet=True)
    print('Descarga terminada.')

print(f'  {ruta_relativa(CSV_COMPLETO)}')
print(f'  Tamaño: {tamano_mb(CSV_COMPLETO):,.1f} MB')
print()


El CSV completo ya existe: no se vuelve a descargar.
  Ingesta_de_Datos/Datasets_raw/andreinovikov_used-cars-dataset/cars.csv
  Tamaño: 138.5 MB



In [6]:
# ── Leer el CSV completo ───────────────────────────────────────────────────

df_completo = pd.read_csv(CSV_COMPLETO, low_memory=False)

print()
print(f'Dataset completo: {df_completo.shape[0]:,} filas, {df_completo.shape[1]} columnas')
print()


Dataset completo: 762,091 filas, 20 columnas



### PASO 4 — Muestra estratificada por marca

El CSV completo pesa más de 100 MB, que es el límite de GitHub. Por eso se toma una **muestra estratificada**:

1. Se calcula cuántas filas le tocan a cada marca según su proporción en el dataset completo (`proporción × 200 000`).
2. Como esas cuotas tienen decimales, cada marca recibe la parte entera y las filas que faltan para llegar exactamente a 200 000 se asignan a las marcas con el decimal más grande (**método del mayor residuo**).
3. Dentro de cada marca, las filas se eligen al azar con `random_state=42`.

> **La muestra no modifica los datos:** cada fila se copia exactamente como viene en el CSV completo, con las mismas columnas y los mismos valores, incluidos nulos y errores. La limpieza se hace en TRANSFORMACION.

In [7]:
# ── Filas que le tocan a cada marca (método del mayor residuo) ─────────────

assert df_completo[COLUMNA_ESTRATO].notna().all(), 'Hay filas sin marca: revisar el estrato'

conteos = df_completo[COLUMNA_ESTRATO].value_counts()
cuota   = conteos / conteos.sum() * TAMANO_MUESTRA          # filas ideales (con decimales)

asignadas = np.floor(cuota).astype(int)                     # parte entera
faltan    = TAMANO_MUESTRA - asignadas.sum()                # filas que faltan para llegar al total
mayores_residuos = (cuota - asignadas).sort_values(ascending=False).index[:faltan]
asignadas[mayores_residuos] += 1

print()
print(f'Marcas: {len(asignadas)}  |  filas asignadas: {asignadas.sum():,}')
print()


Marcas: 30  |  filas asignadas: 200,000



In [8]:
# ── Tomar la muestra ───────────────────────────────────────────────────────

muestra = pd.concat([
    df_completo[df_completo[COLUMNA_ESTRATO] == marca].sample(n=n, random_state=SEMILLA)
    for marca, n in asignadas.items()
]).sort_index()                                             # conserva el orden del archivo original

muestra.to_csv(CSV_MUESTRA, index=False)

print()
print(f'Muestra: {muestra.shape[0]:,} filas, {muestra.shape[1]} columnas')
print(f'Guardada en: {ruta_relativa(CSV_MUESTRA)}  ({tamano_mb(CSV_MUESTRA):,.1f} MB)')
print()


Muestra: 200,000 filas, 20 columnas
Guardada en: Ingesta_de_Datos/Datasets_raw/andreinovikov_used-cars-dataset/cars_muestra.csv  (36.6 MB)



#### Verificación de la muestra

Se comprueba que la muestra **representa** al dataset completo y que **no cambió** los datos.

In [9]:
# ── 1. ¿Cada marca conserva su proporción? ─────────────────────────────────

proporciones = pd.DataFrame({
    'filas_completo': conteos,
    'pct_completo'  : (conteos / conteos.sum() * 100).round(3),
    'filas_muestra' : muestra[COLUMNA_ESTRATO].value_counts(),
})
proporciones['pct_muestra'] = (proporciones['filas_muestra'] / len(muestra) * 100).round(3)
proporciones['diferencia']  = (proporciones['pct_muestra'] - proporciones['pct_completo']).round(3)

print()
print(f"Diferencia máxima de proporción entre marcas: {proporciones['diferencia'].abs().max():.3f} puntos porcentuales")
print()

proporciones


Diferencia máxima de proporción entre marcas: 0.001 puntos porcentuales



,filas_completo,pct_completo,filas_muestra,pct_muestra,diferencia
manufacturer,,,,,
Ford,79526,10.435,20870,10.435,0.000
Toyota,59535,7.812,15624,7.812,0.000
Chevrolet,56043,7.354,14708,7.354,0.000
Nissan,48529,6.368,12736,6.368,0.000
Jeep,41665,5.467,10934,5.467,0.000
Mercedes-Benz,40824,5.357,10714,5.357,0.000
Honda,37612,4.935,9871,4.936,0.001
BMW,37570,4.930,9860,4.930,0.000
Kia,35063,4.601,9202,4.601,0.000


In [10]:
# ── 2. ¿La muestra es una copia fiel (mismas columnas, tipos y valores)? ───

mismas_columnas = list(muestra.columns) == list(df_completo.columns)
mismos_tipos    = (muestra.dtypes == df_completo.dtypes).all()
mismos_valores  = muestra.equals(df_completo.loc[muestra.index])   # cada fila es idéntica a su original

# También se compara la distribución del precio (la variable objetivo)
comparacion_precio = pd.DataFrame({
    'completo': df_completo['price'].describe(),
    'muestra' : muestra['price'].describe(),
}).round(1)

print()
print(f'¿Mismas columnas?               {mismas_columnas}')
print(f'¿Mismos tipos de dato?          {mismos_tipos}')
print(f'¿Filas idénticas a su original? {mismos_valores}')
print()

comparacion_precio


¿Mismas columnas?               True
¿Mismos tipos de dato?          True
¿Filas idénticas a su original? True



,completo,muestra
count,7.620910e+05,200000.0
mean,3.648900e+04,32588.6
std,1.984183e+06,24265.3
min,1.000000e+00,1.0
25%,1.958300e+04,19500.0
50%,2.798900e+04,27990.0
75%,3.948800e+04,39500.0
max,1.000000e+09,2099995.0


In [11]:
# ── Liberar memoria: a partir de aquí solo se usa la muestra ───────────────

del df_completo

print()
print('Dataset completo liberado de memoria')
print()


Dataset completo liberado de memoria



### PASO 5 — Inspección inicial de los datos (Preparación)

La inspección se hace sobre **el CSV de la muestra tal como quedó guardado en disco**, porque eso es lo que se archiva en RAW. Todavía no se modifica nada: solo se conoce el dataset con `inspeccionar_dataset()` de `utils.py`.

In [12]:
# ── Leer la muestra desde el disco e inspeccionarla ────────────────────────

autos = pd.read_csv(CSV_MUESTRA, low_memory=False)

inspeccionar_dataset(autos, nombre=f'{ETIQUETA} — {REF} (muestra)')


  DATASET: autos — andreinovikov/used-cars-dataset (muestra)
  Filas    : 200,000
  Columnas : 20

  Tipos de datos y valores nulos:
    manufacturer                 str          nulos: 0 (0.0%)
    model                        str          nulos: 0 (0.0%)
    year                         int64        nulos: 0 (0.0%)
    mileage                      float64      nulos: 126 (0.1%)
    engine                       str          nulos: 3887 (1.9%)
    transmission                 str          nulos: 2545 (1.3%)
    drivetrain                   str          nulos: 5593 (2.8%)
    fuel_type                    str          nulos: 5977 (3.0%)
    mpg                          str          nulos: 37071 (18.5%)
    exterior_color               str          nulos: 2355 (1.2%)
    interior_color               str          nulos: 15031 (7.5%)
    accidents_or_damage          float64      nulos: 6332 (3.2%)
    one_owner                    float64      nulos: 8265 (4.1%)
    personal_use_only       

,manufacturer,model,year,mileage,engine,transmission,drivetrain,fuel_type,mpg,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,seller_name,seller_rating,driver_rating,driver_reviews_num,price_drop,price
0,Acura,ILX Hybrid 1.5L,2013,92945.0,"1.5L I-4 i-VTEC variable valve control, engine...",Automatic,Front-wheel Drive,Gasoline,39-38,Black,Parchment,0.0,0.0,0.0,Iconic Coach,NaN,4.4,12.0,300.0,13988.0
1,Acura,ILX Hybrid 1.5L,2013,61539.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Silver,NaN,1.0,1.0,1.0,Napleton's Palatine Subaru,4.6,4.4,12.0,NaN,16950.0
2,Acura,NSX Base,2017,15904.0,Twin Turbo Gas/Electric V-6 3.5 L/213,Automatic,All-wheel Drive,Hybrid,21-22,130R White,Ebony,0.0,0.0,1.0,Piazza Acura / Honda of Reading,4.7,4.8,17.0,2111.0,139876.0


### PASO 6 — Almacenamiento en SQLite (capa RAW → Archivado)

Los datos se guardan **sin modificaciones** en la tabla `raw_autos`. Las siguientes capas (EXTRACTED, TRANSFORMED y SERVING) leerán de aquí, siempre respetando la regla de oro:

> **Cada capa lee de la capa inmediatamente anterior. La capa RAW nunca se modifica.**

In [13]:
# ── Guardar la tabla raw_autos ─────────────────────────────────────────────

conn = sqlite3.connect(DB_PATH)

autos.to_sql(TABLA_RAW, conn, if_exists='replace', index=False)

conn.commit()
conn.close()

print()
print(f'{ETIQUETA} → tabla "{TABLA_RAW}" ({autos.shape[0]:,} filas, {autos.shape[1]} columnas)')
print(f'Base de datos: {ruta_relativa(DB_PATH)}  ({tamano_mb(DB_PATH):,.1f} MB)')
print()


autos → tabla "raw_autos" (200,000 filas, 20 columnas)
Base de datos: Ingesta_de_Datos/Base_de_datos/base_datos.db  (35.6 MB)



In [14]:
# ── Verificar las tablas creadas en la base de datos ───────────────────────

listar_tablas(DB_PATH)


  Tablas en: base_datos.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------


  raw_autos                           raw_             200,000     20


['raw_autos']

In [15]:
# ── ¿Lo que quedó en SQLite es igual al CSV de la muestra? ─────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)              # solo lectura
filas_db = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {TABLA_RAW}', conn)['n'][0]
cols_db  = pd.read_sql_query(f'PRAGMA table_info({TABLA_RAW})', conn)['name'].tolist()
conn.close()

print()
print(f'Filas   — CSV: {len(autos):,}  |  SQLite: {filas_db:,}  |  ¿iguales? {len(autos) == filas_db}')
print(f'Columnas — CSV: {autos.shape[1]}  |  SQLite: {len(cols_db)}  |  ¿iguales? {list(autos.columns) == cols_db}')
print()


Filas   — CSV: 200,000  |  SQLite: 200,000  |  ¿iguales? True
Columnas — CSV: 20  |  SQLite: 20  |  ¿iguales? True



### PASO 7 — Consumo: primera consulta sobre los datos archivados

Cerramos el ciclo de vida con un primer **consumo**: una consulta SQL directa sobre la capa RAW, en modo solo lectura.

In [16]:
# ── Anuncios y precio por marca ────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

resumen_marcas = pd.read_sql_query(f'''
    SELECT manufacturer                  AS marca,
           COUNT(*)                      AS anuncios,
           ROUND(AVG(price), 0)          AS precio_promedio,
           MIN(price)                    AS precio_min,
           MAX(price)                    AS precio_max,
           ROUND(AVG(mileage), 0)        AS millas_promedio
    FROM {TABLA_RAW}
    GROUP BY manufacturer
    ORDER BY anuncios DESC
''', conn)

conn.close()

resumen_marcas

,marca,anuncios,precio_promedio,precio_min,precio_max,millas_promedio
0,Ford,20870,33676.0,1295.0,1295000.0,60287.0
1,Toyota,15624,30017.0,1663.0,295800.0,63548.0
2,Chevrolet,14708,31770.0,1900.0,329995.0,60695.0
3,Nissan,12736,23447.0,750.0,274496.0,56434.0
4,Jeep,10934,31429.0,1600.0,199991.0,51896.0
5,Mercedes-Benz,10714,47838.0,2295.0,1150000.0,44635.0
6,Honda,9871,24907.0,2000.0,85443.0,61159.0
7,BMW,9860,38089.0,1995.0,1750000.0,49732.0
8,Kia,9202,22669.0,1492.0,66300.0,51355.0
9,GMC,7758,40310.0,2595.0,209999.0,62363.0


> **Lectura:** los precios mínimos y máximos muestran valores imposibles para un anuncio real, como precios de 1 USD o de millones. RAW se queda como llegó: esos casos se tratan en TRANSFORMACION.

**Siguiente capa:** `EXTRACCION.ipynb` lee `raw_autos` y genera `extracted_autos`.